# Solving MaxCut Problem Using Quantum Approximate Optimization Algorithm

The Quantum Approximate Optimization Algorithm (QAOA) is a hybrid quantum-classical algorithm for finding near-optimal solutions for combinatorial optimization problems. This algorithm was designed for NISQ-era quantum devices, which suffer from high noise rates and thus cannot run deep circuits. Instead, the algorithm combines results of running multiple shallow circuits with classical optimization of the parameters of these circuits.

This kata walks you through implementing QAOA for the MaxCut problem - the problem of dividing the vertices of the graph into two groups to maximize the number of edges that connect vertices from different groups.
We will start with a simple instance of MaxCut: the problem of finding bit strings of given length that consist of alternating 0s and 1s.

**This kata covers the following topics**:

- Defining optimization problems on graphs
- Quantum Approximate Optimization Algorithm (QAOA) and its building blocks
- Using QAOA to solve the MaxCut problem

**What you should know to start working on this kata**:

- Fundamental quantum concepts, especially rotation gates
- Basics of classical optimization libraries

In [ ]:
# Execute this cell to prepare the test infrastructure
from psiqdk.workbench import QPU, Qubits
from test_MaxCutQAOA import problem

## Defining optimization problems on graphs

The MaxCut problem we consider in this kata is an example of _quadratic unconstrained binary optimization (QUBO) problem_. QUBO problems involve optimizing a quadratic objective function with binary variables without additional constraints. What does this mean, and how can we map such a problem to a graph?

In a QUBO problem, you aim to minimize the _cost function_ $C(x)$ of $N$ Boolean variables: $x \in \{0, 1\}^N$. The cost function is quadratic: it is defined as a linear combination of terms $x_j$ and $x_jx_k$ with fixed coefficients:

$$C(x) = \sum_{j=0}^{N-1} a_j x_j + \sum_{j,k=0}^{N-1} b_{jk} x_j x_k$$

A QUBO problem can be mapped to a graph and vice versa. In such a mapping, the Boolean variables $x_j$ correspond to labels of graph vertices, coefficients $a_j$ - weights of vertices, and coefficients $b_{jk}$ - weights of edges connecting vertices.

### Example: MaxCut problem

MaxCut problem is formulated as follows: given an undirected graph with vertices $V$ and edges $E$, find a "cut" (a split of vertices in two groups) that _maximizes_ the number of edges being "cut", that is, the number of edges that connect vertices from different groups. How can we convert this problem to a QUBO formulation?

1. We will label each vertex $x_j$ as $0$ or $1$ to indicate which group it belongs to.
2. A "cut" that _maximizes_ the number of "cut" edges also _minimizes_ the number of "uncut" edges, that is, edges that connect vertices from the same group. We will define the cost function $C(x)$ as the number of "uncut" edges, and will aim to minimize it.
3. An edge connecting vertices $j$ and $k$ is "uncut" if $x_j = x_k$. Remembering that $x_j \in \{0, 1\}$, we can write the cost function as follows:

   $$C(x) = \sum_{jk \in E} (x_j = x_k) = \sum_{jk \in E} (x_j x_k + (1-x_j)(1-x_k)) = \sum_{jk \in E} (2 x_j x_k - x_j - x_k + 1)$$

   You can see that indeed the cost function is quadratic, and this formulation leads us to a QUBO problem.

### A simpler example: finding bit strings with alternating bits

To start with, we'll look at an even simpler problem which can be rephrased as a special case of MaxCut problem: given an integer $N$, find an $N$-bit string that consists of alternating bits $0$ and $1$.

Of course, this is a very simple problem! You can tell right away what the answers are: the two bit strings $0101...$ and $1010...$. However, it will be useful as a starting point for implementing the QAOA algorithm and exploring its behavior.

The graph representation for this problem can be formulated as follows. The $j$-th bit of the string corresponds to vertex with index $j$, and the edges connect vertices $j$ and $j+1$ for all $j$ from $0$ to $N-2$, inclusive. The cost function then looks as follows:

$$C(x) = \sum_{j = 0}^{N-2} (x_j = x_{j+1})$$

This graph is bipartite, so the perfect solution leaves no edge uncut, achieving $C(x) = 0$. Let's see if QAOA algorithm manages to find that solution!

## Quantum approximate optimization algorithm

### Inputs

The main input to QAOA is the cost function which defines the problem. Unlike Grover's search, this cost function is not a black box: we will rely on its structure to implement the steps of the algorithm.

### Algorithm outline

QAOA is a hybrid quantum-classical algorithm. This means that it consists of two parts: the parameterized quantum circuit that runs on a quantum device and the classical routine that uses the results of circuit execution to optimize the circuit parameters.

The quantum circuit used in QAOA consists of $K$ layers that use $2K$ parameters $\beta_1, \gamma_1, ..., \beta_K, \gamma_K$.
The circuit looks as follows:

<center><img src="./media/oaoa_circuit.png" width="70%"/></center>

1. Prepare an equal superposition of all $N$-qubit basis states.
2. Apply the phase separation unitary, also known as the cost unitary, parameterized with $\gamma$: 
   $$U_C(\gamma)|x\rangle = e^{-i\gamma C(x)}|x\rangle$$
   The effect is similar to applying the phase oracle in Grover's search algorithm: the relative phases of basis states change depending on the value of the cost function for them.
3. Apply the mixer unitary, parameterized with $\beta$:
   $$U_B(\beta) \ket{x} = e^{-i\beta \sum_{j=0}^{N-1} X_j} \ket{x}$$
   The effect is similar to reflection about the mean in Grover's search algorithm: this unitary does not depend on the problem definition and serves to "mix up" the basis states.
4. Repeat steps 2 and 3 $K$ times, using parameters $\beta_j$ and $\gamma_j$ as parameters in repetition number $j$.
5. Measure the resulting bit string to get a bit string $x$.

The classical routine optimizes the parameters $\beta, \gamma$ using the following procedure:

1. Run the parameterized circuit for a fixed value of $\beta, \gamma$, calculate the cost $C(x)$ for each run result $x$ and estimate the average cost per run.
2. Use a classical optimizer to find the values of $\beta^{(opt)}, \gamma^{(opt)}$ that minimize the average cost.

### Algorithm results

Once the algorithm found the optimal parameter values $\beta^{(opt)}, \gamma^{(opt)}$, the last step of the algorithm uses these values to run the parameterized quantum circuit again. This time, its output is considered to be the problem solution.

### Further reading

You can read more about the details of QAOA in [this blog post](https://www.mustythoughts.com/quantum-approximate-optimization-algorithm-explained).

## Implementing QAOA for finding alternating bit strings

In the next section of this kata, you'll practice implementing QAOA for solving our example problem: finding an $N$-bit string that consists of alternating bits $0$ and $1$.

## Problem 1. Classical cost function

**Input**: A list of integers $x$, each of them 0 or 1, representing the bits of the string.

**Output**: The value of the cost function for the problem of finding the bit strings consisting of alternating bits. As described above, we define the cost function as the number of pairs of identical bits in adjacent positions:

$$C(x) = \sum_{j = 0}^{N-2} (x_j = x_{j+1})$$

In [ ]:
@problem
def classical_cost(x: list[int]) -> int:
    # Write your code here
    ...

## Problem 2. Phase separation unitary

**Inputs:**
1. A `Qubits` register of length $N$.
2. A real value of the parameter $\gamma$, in radians.

**Goal:** 
Implement the first unitary used in QAOA - the phase separation unitary. This unitary should implement the transformation

$$U_C(\gamma)|x\rangle = e^{-i\gamma C(x)}|x\rangle$$

<details>
<summary><strong>Need a hint?</strong></summary>

The cost function can be expressed as a sum of terms applied to adjacent pairs of qubits. This allows you to decompose the unitary $U_C(\gamma)$ into a sequence of 1- and 2-qubit gates without computing $C(x)$ explicitly.
</details>

In [ ]:
@problem
def phase_separation_unitary(reg: Qubits, gamma: float) -> None:
    # Write your code here
    ...

## Problem 3. The mixer unitary

**Inputs:**
1. A `Qubits` register of length $N$.
2. A real value of the parameter $\beta$, in radians.

**Goal:** 
Implement the second unitary used in QAOA - the mixer. This unitary should implement the transformation
$$U_B(\beta) \ket{x} = e^{-i\beta \sum_{j=0}^{N-1} X_j} \ket{x}$$
Here $X_j$ is the Pauli X gate acting on qubit with index $j$. By definition,
$$e^{i\theta X} = \begin{bmatrix} \cos(\theta) & i\sin(\theta) \\ i\sin(\theta) & cos(\theta) \end{bmatrix}$$

<details>
<summary><strong>Need a hint?</strong></summary>

Does any of the rotation gates have a similar matrix?
</details>

In [ ]:
@problem
def mixer_unitary(reg: Qubits, beta: float) -> None:
    # Write your code here
    ...

## Demo: Depth 1 QAOA to find alternating bit strings

This demo shows an implementation of depth $1$ QAOA (the number of layers $K = 1$) that solves the problem of finding bit strings that consist of alternating bits for the string length $n$ from $2$ to $6$.

In [ ]:
from functools import partial
from psiqdk.workbench import QPU, Qubits
from scipy.optimize import minimize

def run_qaoa_circuit(n: int, gamma: float, beta: float) -> list[int]:
    '''Run depth 1 QAOA circuit parameterized by gamma and beta and return the measurement results'''

    from psiqdk.workbench import units
    def phase_separation_unitary(reg: Qubits, gamma: float) -> None:
        for j in range(len(reg) - 1):
            reg[j].phase(-gamma * units.rad, cond=reg[j + 1])
            reg[j].x()
            reg[j].phase(-gamma * units.rad, cond=~reg[j + 1])
            reg[j].x()
    def mixer_unitary(reg: Qubits, beta: float) -> None:
        for j in range(len(reg)):
            reg[j].rx(2 * beta * units.rad)

    # Create QPU and allocate N qubits.
    qpu = QPU(num_qubits=n)
    reg = Qubits(n, "reg", qpu)
    # Prepare an equal superposition of all basis states.
    reg.had()
    # Apply the phase separation unitary with parameter gamma.
    phase_separation_unitary(reg, gamma)
    # Apply the mixer unitary with parameter beta.
    mixer_unitary(reg, beta)
    # Measure all qubits and return the results.
    return [q.read() for q in reg]


def classical_cost(x: list[int]) -> int:
    '''Calculate classical cost function for a bit string'''
    return sum([1 if x[j] == x[j + 1] else 0 for j in range(len(x) - 1)])


def answer_frequency(n: int, parameters: list[float]) -> dict:
    '''Get frequencies of results produced by the circuit running with fixed parameters.'''
    shots = 100
    dict = {}
    results = [run_qaoa_circuit(n, parameters[0], parameters[1]) for _ in range(shots)]
    for res in results:
        key = ''.join(map(str, res))
        if key in dict:
            dict[key] += 1
        else:
            dict[key] = 1
    return {k : (v / shots) for k, v in dict.items()}


def find_alternating_bitstrings(n: int) -> None:
    '''Find bit strings of length n that consist of alternating bits using QAOA.'''

    def multishot_cost(parameters: list[float]) -> float:
        '''Estimate expectation of classical cost function across multiple runs of the circuit with fixed parameters.'''
        shots = 50
        results = [run_qaoa_circuit(n, parameters[0], parameters[1]) for _ in range(shots)]
        return sum(classical_cost(res) for res in results) / shots

    # Learn the optimal parameters
    initial_parameters = [1.5, 1]
    out = minimize(multishot_cost, initial_parameters, method="COBYLA", options={'maxiter' : 100})
    optimal_parameters = out.x
    print(f"Optimal Values for n = {n}: γ = {optimal_parameters[0]:.2f}, β = {optimal_parameters[1]:.2f}")

    # Use the optimal parameters to estimate the probability of finding the correct solution
    ans = answer_frequency(n, optimal_parameters)
    print(sorted(ans.items(), key=lambda x:x[1], reverse=True))
    success_probability = ans.get('01010101'[:n], 0) + ans.get('10101010'[:n], 0)
    print(f"Success probability: {success_probability:.2f}")
    print()

for n in range(2, 7):
    find_alternating_bitstrings(n)

You can see that, while for the small problem instance success probability is close to $1$, it falls quickly as the problem size increases.
If you run the demo several times, you'll also notice that the results can vary dramatically from run to run. 
This is caused by the probabilistic nature of cost function evaluation: the randomness of measurement outcomes impacts the estimated cost for the fixed parameter value, which can lead the classical optimizer in a completely wrong direction.

To get better results, you need to add more layers to the circuit, but that increases the number of values of $\gamma$ and $\beta$ the algorithm needs to learn, which makes the optimization problem of finding the best parameter values harder.

## Conclusion

Congratulations! In this kata you learned the basics of quantum approximate optimization algorithm.

> Copyright (c) 2026 PsiQuantum